# Feature Visualization Pipeline — Music Genre Classification

This notebook demonstrates the audio feature extraction pipeline used in our
music genre classification system. We visualize each step of the transformation
process from raw features to the final classification.

## Pipeline: Feature Vector → Analysis → Visualization

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add ml directory to path
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'ml'))
from core.preprocessing import load_raw_dataframe, GENRE_LABELS
from core.features import FEATURE_GROUPS, FEATURES_TO_USE, FEATURES_TO_EXCLUDE

# Load dataset
df = pd.read_csv('../Data/features_3_sec.csv')
print(f'Dataset: {df.shape[0]} samples, {df.shape[1]} columns')
print(f'Genres: {sorted(df["label"].unique())}')
df.head()

## 1. Feature Pipeline Overview

The audio-to-features pipeline works as follows:

```
Raw Audio (.wav)  →  librosa.load()  →  Waveform (time domain)
     ↓
Short-Time Fourier Transform (STFT)  →  Spectrogram (time-frequency)
     ↓
Mel Filterbank Application  →  Mel-Spectrogram
     ↓
Discrete Cosine Transform (DCT)  →  MFCCs (13-20 coefficients)
     ↓
Combined with Spectral Features  →  Feature Vector (57 dimensions)
```

Since we work with pre-extracted features from CSV, this notebook visualizes
the statistical properties of these features across genres.

In [ ]:
# Step 1: Feature Group Statistics
print('Feature Groups in our dataset:\n')
for name, group in FEATURE_GROUPS.items():
    cols = [c for c in group['columns'] if c in df.columns]
    print(f'{name:25s} | {len(cols):2d} features | {group["description"]}')

## 2. MFCC Analysis Across Genres

MFCCs are the most important features. Let's visualize how they differ across genres.

In [ ]:
# MFCC mean values for each genre
mfcc_cols = [f'mfcc{i}_mean' for i in range(1, 21)]
genres_to_compare = ['classical', 'metal', 'jazz', 'rock', 'hiphop']

fig, ax = plt.subplots(figsize=(14, 6))
for genre in genres_to_compare:
    genre_data = df[df['label'] == genre][mfcc_cols].mean()
    ax.plot(range(1, 21), genre_data, marker='o', linewidth=2, label=genre)

ax.set_xlabel('MFCC Coefficient Number', fontsize=12)
ax.set_ylabel('Mean Value', fontsize=12)
ax.set_title('MFCC Profile Comparison Across Genres', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(range(1, 21))
plt.tight_layout()
plt.show()

In [ ]:
# MFCC heatmap across all genres
mfcc_means = df.groupby('label')[mfcc_cols].mean()
fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(mfcc_means, cmap='RdBu_r', center=0, annot=False,
            xticklabels=[f'MFCC{i}' for i in range(1, 21)],
            ax=ax)
ax.set_title('Mean MFCC Values per Genre (Heatmap)', fontsize=14, fontweight='bold')
ax.set_ylabel('Genre')
plt.tight_layout()
plt.show()

print('\nKey observations:')
print('- Classical has distinctly different MFCC1 values (orchestral timbre)')
print('- Metal shows unique MFCC patterns due to distorted guitar')
print('- Rock and country overlap significantly (similar instruments)')

## 3. Spectral Features: Centroid, ZCR, Bandwidth

In [ ]:
# Box plots for key spectral features
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

features = ['spectral_centroid_mean', 'zero_crossing_rate_mean', 'spectral_bandwidth_mean']
titles = ['Spectral Centroid (Brightness)', 'Zero Crossing Rate (Noisiness)', 'Spectral Bandwidth (Spread)']

for ax, feat, title in zip(axes, features, titles):
    genre_order = df.groupby('label')[feat].median().sort_values().index
    sns.boxplot(data=df, x='label', y=feat, ax=ax, order=genre_order,
                palette='Set3')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Key Spectral Features by Genre', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. t-SNE: 2D Embedding of Features

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import MinMaxScaler

# Sample for speed
sample = df.sample(2000, random_state=42)
feature_cols = [c for c in sample.columns if c not in ['filename', 'length', 'label']]
X = MinMaxScaler().fit_transform(sample[feature_cols].values)

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_2d = tsne.fit_transform(X)

fig, ax = plt.subplots(figsize=(12, 8))
for genre in sorted(sample['label'].unique()):
    mask = sample['label'].values == genre
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], label=genre, alpha=0.6, s=20)

ax.set_title('t-SNE Visualization of Genre Feature Space', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, markerscale=2)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print('Notice: Classical and metal form distinct clusters (easy to classify)')
print('Rock, country, and blues overlap significantly (harder to distinguish)')

## 5. Feature Importance (Mutual Information)

In [ ]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

feature_cols = [c for c in df.columns if c not in ['filename', 'length', 'label']]
X = df[feature_cols].values
y = LabelEncoder().fit_transform(df['label'].values)

mi = mutual_info_classif(X, y, random_state=42)
mi_series = pd.Series(mi, index=feature_cols).sort_values(ascending=True)

# Color by feature group
colors = []
for feat in mi_series.index:
    if 'mfcc' in feat:
        colors.append('#2196F3')
    elif 'centroid' in feat:
        colors.append('#4CAF50')
    elif 'zero_crossing' in feat:
        colors.append('#FF9800')
    elif feat in ['tempo', 'harmony_mean', 'harmony_var', 'perceptr_mean', 'perceptr_var', 'rms_mean', 'rms_var']:
        colors.append('#F44336')
    else:
        colors.append('#9E9E9E')

fig, ax = plt.subplots(figsize=(12, 12))
ax.barh(range(len(mi_series)), mi_series.values, color=colors)
ax.set_yticks(range(len(mi_series)))
ax.set_yticklabels(mi_series.index, fontsize=8)
ax.set_xlabel('Mutual Information Score', fontsize=12)
ax.set_title('Feature Importance — Mutual Information with Genre Label', fontsize=14, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2196F3', label='MFCC (USED)'),
    Patch(facecolor='#4CAF50', label='Spectral Centroid (USED)'),
    Patch(facecolor='#FF9800', label='Zero Crossing Rate (USED)'),
    Patch(facecolor='#F44336', label='EXCLUDED features'),
    Patch(facecolor='#9E9E9E', label='Other features'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

## 6. Correlation Analysis

In [ ]:
# Top features correlation matrix
top_features = [
    'mfcc1_mean', 'mfcc2_mean', 'mfcc3_mean', 'mfcc4_mean', 'mfcc5_mean',
    'spectral_centroid_mean', 'zero_crossing_rate_mean',
    'spectral_bandwidth_mean', 'rolloff_mean', 'chroma_stft_mean',
    'rms_mean', 'tempo'
]
corr = df[top_features].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
            square=True, ax=ax)
ax.set_title('Feature Correlation Matrix (Top Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Key observations:')
print('- Spectral centroid and rolloff are highly correlated (0.9+) — expected, both measure brightness')
print('- MFCCs are mostly independent of each other — good for classification')
print('- Tempo has low correlation with all features — confirms its low discriminative power')

## 7. Genre Similarity Analysis

In [ ]:
# Genre distance matrix using feature centroids
from scipy.spatial.distance import cdist

genre_centroids = df.groupby('label')[feature_cols].mean()
dist_matrix = cdist(genre_centroids.values, genre_centroids.values, metric='euclidean')
dist_df = pd.DataFrame(dist_matrix, index=genre_centroids.index, columns=genre_centroids.index)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(dist_df, annot=True, fmt='.0f', cmap='YlOrRd_r', ax=ax)
ax.set_title('Genre Distance Matrix (Euclidean Distance in Feature Space)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nClosest genre pairs (most confusable):')
np.fill_diagonal(dist_matrix, np.inf)
for _ in range(5):
    idx = np.unravel_index(dist_matrix.argmin(), dist_matrix.shape)
    g1, g2 = genre_centroids.index[idx[0]], genre_centroids.index[idx[1]]
    print(f'  {g1} ↔ {g2}: distance = {dist_matrix[idx]:.1f}')
    dist_matrix[idx] = np.inf
    dist_matrix[idx[1], idx[0]] = np.inf

## Summary

### Key Findings:
1. **MFCCs are dominant** — they capture the most genre-discriminative information
2. **Classical and metal are easiest** to classify (distinct spectral profiles)
3. **Rock and country are hardest** (similar acoustic instrumentation)
4. **Tempo is unreliable** — high variance within genres
5. **Feature space is well-structured** — t-SNE shows clear clusters for most genres